# IndoBERT Encode — ta-news-rec (Eksperimen 2)

Langkah:
1. Menu Runtime > Change runtime type > pilih **T4 GPU** (JANGAN TPU).
2. Run All dari atas ke bawah.
3. Saat diminta upload: pilih file `news_processed.parquet` dari repo lokal (`data/processed/`).
4. Tunggu encode selesai (±10–20 menit), verifikasi shape (93698, 768) + norm ~1.0.
5. File `indobert_embeddings.npy` (~287 MB) otomatis terdownload. Copy ke repo lokal di `models/indobert_embeddings.npy`, lalu jalankan `venv/bin/python src/evaluation/indobert_eval.py`.

Aturan penting: teks = title + abstract MENTAH (bukan processed_text). Urutan baris parquet dipertahankan persis — itu mapping nid.

In [ ]:
!nvidia-smi
import torch
print('cuda:', torch.cuda.is_available(), '| gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')

In [ ]:
!pip install -q sentence-transformers

In [ ]:
from google.colab import files
print('Upload file news_processed.parquet dari data/processed/ repo lokal:')
uploaded = files.upload()
print(list(uploaded.keys()))

In [ ]:
import glob, json
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

MODEL = 'firqaaa/indo-sentence-bert-base'
pq = glob.glob('/content/*.parquet')[0]
print('parquet:', pq)
df = pd.read_parquet(pq).reset_index(drop=True)
print('rows:', len(df), '| cols:', list(df.columns))
assert len(df) == 93698, f'row count beda: {len(df)}'
texts = (df['title'].fillna('') + ' ' + df['abstract'].fillna('')).str.strip().tolist()
assert all(texts), 'ada teks kosong'

model = SentenceTransformer(MODEL)  # otomatis pakai GPU kalau runtime T4
print('dim:', model.get_sentence_embedding_dimension(), '| device:', model.device)

emb = model.encode(texts, batch_size=128, show_progress_bar=True,
                     convert_to_numpy=True, normalize_embeddings=True).astype(np.float32)
print('shape:', emb.shape)
norms = np.linalg.norm(emb, axis=1)
print(f'norm mean={norms.mean():.4f} min={norms.min():.4f} max={norms.max():.4f}')
assert emb.shape == (93698, 768) and abs(norms.mean() - 1.0) < 1e-3

np.save('/content/indobert_embeddings.npy', emb)
meta = {'model': MODEL, 'n_news': len(texts), 'dim': 768, 'dtype': 'float32',
        'l2_normalized': True, 'text_source': 'title + abstract MENTAH',
        'row_order': 'sejajar news_processed.parquet (reset_index)',
        'size_MB': round(emb.nbytes / 1e6, 1)}
open('/content/indobert_meta.json', 'w').write(json.dumps(meta, indent=2))
print(json.dumps(meta, indent=2))
print('Tersimpan: /content/indobert_embeddings.npy')

In [ ]:
from google.colab import files
files.download('/content/indobert_embeddings.npy')
files.download('/content/indobert_meta.json')